# Benchmark LLM pour TontoumaBot

**Objectif** : choisir le meilleur générateur pour le RAG.
**Protocole** : contexte FIGÉ, seul le LLM varie — on évalue le générateur, pas le retriever.

Configuré pour Google Colab gratuit, GPU T4 (16 Go VRAM). Conséquence matérielle : `Qwen2.5-7B-Instruct` ne tient pas en fp16 sur une T4 (~15 Go de poids + activations + cache KV > 16 Go) — il est donc chargé en 4-bit NF4 (~4,7 Go). C'est déclaré explicitement dans les résultats via la colonne `precision`, à mentionner dans ton mémoire.

⚠️ **Contrainte mémoire importante** : les modèles sont chargés et évalués **un par un, puis déchargés** avant de passer au suivant — impossible de garder 6 LLM en mémoire simultanément sur une T4. C'est géré automatiquement dans ce notebook (Cellule 4).

## Cellule 1 — Installation

In [ ]:
!pip install -q -U transformers accelerate sentencepiece
!pip install -q -U "protobuf>=5.27.0,<6"
!pip install -q -U bitsandbytes
!pip install -q evaluate sacrebleu jiwer pandas matplotlib

# --> REDÉMARRE LA SESSION après cette cellule (Runtime -> Restart runtime),
#     puis exécute la suite (Runtime -> Run all) sans revenir sur celle-ci.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Cellule 2 — Imports et configuration

In [ ]:
import gc
import numpy as np
import json
import re
import time
import unicodedata
import warnings
from pathlib import Path

import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")


# Catalogue des LLM candidats
#   type  : "seq2seq" (encodeur-décodeur) ou "causal" (décodeur seul)
#   quant : "fp16" | "8bit" | "4bit"  -> impose la précision de chargement
#   ctx   : fenêtre de contexte utilisable
LLM_CANDIDATS = {
    "flan-t5-base": {
        "id": "google/flan-t5-base", "type": "seq2seq", "ctx": 512,
        "params_b": 0.25, "quant": "fp16", "vram_estimee_go": 0.6, "role": "baseline",
    },
    "mt0-base": {
        "id": "bigscience/mt0-base", "type": "seq2seq", "ctx": 1024,
        "params_b": 0.58, "quant": "fp16", "vram_estimee_go": 1.3, "role": "baseline multilingue",
    },
    "qwen2.5-1.5b": {
        "id": "Qwen/Qwen2.5-1.5B-Instruct", "type": "causal", "ctx": 8192,
        "params_b": 1.5, "quant": "fp16", "vram_estimee_go": 3.5, "role": "candidat léger",
    },
    "qwen2.5-3b": {
        "id": "Qwen/Qwen2.5-3B-Instruct", "type": "causal", "ctx": 8192,
        "params_b": 3.0, "quant": "fp16", "vram_estimee_go": 7.0, "role": "candidat principal",
    },
    "qwen2.5-7b": {
        "id": "Qwen/Qwen2.5-7B-Instruct", "type": "causal", "ctx": 8192,
        "params_b": 7.6, "quant": "4bit", "vram_estimee_go": 5.5, "role": "candidat principal (ton modèle)",
    },
    # "qwen2.5-7b-8bit": {
    #     "id": "Qwen/Qwen2.5-7B-Instruct", "type": "causal", "ctx": 8192,
    #     "params_b": 7.6, "quant": "8bit", "vram_estimee_go": 9.0, "role": "étude de la quantification",
    # },
}

MAX_NEW_TOKENS = 220
NUM_BEAMS = 1  # 1 = greedy, déterministe et rapide

VRAM_TOTALE_GO = (
    torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0
)
MARGE_SECURITE_GO = 1.5

RESULTS_DIR = Path("resultats_llm")
RESULTS_DIR.mkdir(exist_ok=True)
MEILLEUR_LLM_DIR = Path("meilleur_llm")

print("\nFaisabilité sur ce matériel :")
for nom, cfg in LLM_CANDIDATS.items():
    besoin = cfg["vram_estimee_go"]
    if DEVICE != "cuda":
        verdict = "CPU : lent mais possible" if cfg["params_b"] < 1 else "CPU : trop lent"
    elif besoin + MARGE_SECURITE_GO <= VRAM_TOTALE_GO:
        verdict = "OK"
    else:
        verdict = "RISQUE d'OOM"
    print(f"  {nom:16s} {cfg['quant']:5s} ~{besoin:4.1f} Go  -> {verdict}")


## Cellule 3 — Jeu de test avec contexte FIGÉ

Règle d'or : les passages sont identiques pour tous les LLM. On évalue le générateur, pas le retriever.

4 catégories obligatoires :
- **répondable** : la réponse est dans le contexte
- **multi_passage** : la réponse nécessite 2 passages
- **absente** : PIÈGE, la réponse n'est PAS dans le contexte
- **contradictoire** : PIÈGE, deux passages se contredisent

In [ ]:
JEU_TEST = [
    {
        "id": 1, 
        "categorie": "repondable",
        "question": "Quels documents faut-il fournir pour obtenir une carte nationale d'identite ?",
        "passages": [
            {"source": "guide_cni.pdf", "page": 3, "texte": (
                "Pour la demande d'une carte nationale d'identite, le demandeur "
                "doit presenter un extrait de naissance de moins de trois mois, "
                "un certificat de nationalite senegalaise, deux photos d'identite "
                "recentes et une piece justificative de domicile."
            )},
            {"source": "guide_cni.pdf", "page": 4, "texte": (
                "Le depot du dossier se fait au centre d'etat civil du lieu de "
                "residence. Le delai de traitement est de vingt et un jours ouvrables."
            )},
        ],
        "reponse_attendue": (
            "Un extrait de naissance de moins de trois mois, un certificat de "
            "nationalite senegalaise, deux photos d'identite recentes et un "
            "justificatif de domicile."
        ),
        "doit_repondre": True,
    },
    {
        "id": 2, "categorie": "repondable",
        "question": "Quel est le delai de traitement d'une demande de carte nationale ?",
        "passages": [
            {"source": "guide_cni.pdf", "page": 4, "texte": (
                "Le depot du dossier se fait au centre d'etat civil du lieu de "
                "residence. Le delai de traitement est de vingt et un jours ouvrables."
            )},
        ],
        "reponse_attendue": "Vingt et un jours ouvrables.",
        "doit_repondre": True,
    },
    {
        "id": 3, "categorie": "multi_passage",
        "question": "Ou deposer le dossier et combien coute la demande ?",
        "passages": [
            {"source": "guide_cni.pdf", "page": 4, "texte": "Le depot du dossier se fait au centre d'etat civil du lieu de residence."},
            {"source": "tarifs_2026.pdf", "page": 1, "texte": (
                "Le timbre fiscal pour une premiere demande de carte nationale "
                "d'identite est fixe a mille francs CFA. Le renouvellement est "
                "facture deux mille francs CFA."
            )},
        ],
        "reponse_attendue": "Au centre d'etat civil du lieu de residence, pour mille francs CFA en premiere demande.",
        "doit_repondre": True,
    },
    {
        "id": 4, "categorie": "absente",
        "question": "Quel est le montant de l'amende en cas de perte de la carte nationale ?",
        "passages": [
            {"source": "guide_cni.pdf", "page": 3, "texte": "Pour la demande d'une carte nationale d'identite, le demandeur doit presenter un extrait de naissance de moins de trois mois."},
        ],
        "reponse_attendue": "INFORMATION_ABSENTE",
        "doit_repondre": False,
    },
    {
        "id": 5, "categorie": "absente",
        "question": "Quels sont les horaires d'ouverture du centre d'etat civil de Dakar ?",
        "passages": [
            {"source": "tarifs_2026.pdf", "page": 1, "texte": "Le timbre fiscal pour une premiere demande de carte nationale d'identite est fixe a mille francs CFA."},
        ],
        "reponse_attendue": "INFORMATION_ABSENTE",
        "doit_repondre": False,
    },
    {
        "id": 6, "categorie": "contradictoire",
        "question": "Combien de photos d'identite faut-il fournir ?",
        "passages": [
            {"source": "guide_cni.pdf", "page": 3, "texte": "Le demandeur doit fournir deux photos d'identite recentes."},
            {"source": "circulaire_2025.pdf", "page": 2, "texte": "A compter de janvier 2025, le nombre de photos exige est porte a quatre pour toute nouvelle demande."},
        ],
        "reponse_attendue": "Les documents se contredisent : deux photos selon le guide, quatre selon la circulaire de 2025.",
        "doit_repondre": True,
    },
]

# --> AJOUTE tes propres cas : vise 30 a 50 questions au total,
#     avec au moins 20% de cas "absente".

df_test = pd.DataFrame(
    [{"id": c["id"], "categorie": c["categorie"], "question": c["question"]} for c in JEU_TEST]
)
print(f"\n{len(JEU_TEST)} cas de test")
print(df_test["categorie"].value_counts().to_string())

## Cellule 4 — Construction du prompt (contexte figé)

Même template pour tous les modèles, avec consigne explicite d'abstention et de signalement des contradictions — c'est ce qui rend la comparaison valide.

In [ ]:
ROLE_SYSTEME = (
    "Tu es TontoumaBot, l'assistant administratif officiel. "
    "Réponds UNIQUEMENT à partir des passages fournis ci-dessous."
)

CONSIGNE = (
    "Règles strictes :\n"
    "1. Si la réponse n'est pas dans les passages, réponds EXACTEMENT : INFORMATION_ABSENTE\n"
    "2. Si les passages se contredisent, signale-le explicitement en citant les deux valeurs et leurs sources.\n"
    "3. Ne jamais inventer d'information absente des passages.\n"
    "4. Réponse courte et directe, en français."
)


def formater_passages(passages):
    return "\n\n".join(
        f"[Source : {p['source']}, page {p['page']}]\n{p['texte']}" for p in passages
    )


def construire_prompt(question, passages):
    contexte = formater_passages(passages)
    return f"{ROLE_SYSTEME}\n\n{contexte}\n\n{CONSIGNE}\n\nQuestion : {question}\nRéponse :"


# Vérification rapide sur le cas 6 (contradictoire)
print(construire_prompt(JEU_TEST[5]["question"], JEU_TEST[5]["passages"]))

## Cellule 5 — Chargement / déchargement mémoire-safe d'un LLM

Sur une T4, impossible de garder plusieurs LLM en mémoire en même temps. Cette fonction charge un modèle avec la quantification demandée par `LLM_CANDIDATS`, et une fonction sœur le décharge proprement avant de passer au suivant.

In [ ]:
def charger_llm(nom_modele):
    cfg = LLM_CANDIDATS[nom_modele]
    kwargs = {}

    if cfg["quant"] == "4bit":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
        )
    elif cfg["quant"] == "8bit":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    else:
        kwargs["torch_dtype"] = DTYPE

    tokenizer = AutoTokenizer.from_pretrained(cfg["id"])

    if cfg["type"] == "seq2seq":
        model = AutoModelForSeq2SeqLM.from_pretrained(cfg["id"], device_map="auto", **kwargs)
    else:
        model = AutoModelForCausalLM.from_pretrained(cfg["id"], device_map="auto", **kwargs)

    return tokenizer, model


def decharger_llm(tokenizer, model):
    del tokenizer, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def generer_reponse(tokenizer, model, cfg, prompt):
    t0 = time.perf_counter()

    if cfg["type"] == "seq2seq":
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=cfg["ctx"]).to(model.device)
        sortie = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
        texte = tokenizer.decode(sortie[0], skip_special_tokens=True)
    else:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
        sortie = model.generate(
            inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
        texte = tokenizer.decode(sortie[0][inputs.shape[-1]:], skip_special_tokens=True)

    temps_ms = (time.perf_counter() - t0) * 1000
    return texte.strip(), temps_ms


print("Fonctions de chargement/génération prêtes.")

## Cellule 6 — Métriques d'évaluation

- **Répondable / multi_passage** : F1 au niveau des mots avec la réponse attendue
- **Absente** : le modèle a-t-il correctement dit `INFORMATION_ABSENTE` (abstention) ?
- **Contradictoire** : le modèle a-t-il bien signalé la contradiction ?

In [ ]:
def normaliser(texte):
    texte = str(texte).lower().strip()
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode("utf-8")
    texte = re.sub(r"[^\w\s]", " ", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte


def f1_mots(reference, hypothese):
    mots_ref = normaliser(reference).split()
    mots_hyp = normaliser(hypothese).split()
    if not mots_ref or not mots_hyp:
        return 0.0
    communs = set(mots_ref) & set(mots_hyp)
    if not communs:
        return 0.0
    precision = len(communs) / len(set(mots_hyp))
    rappel = len(communs) / len(set(mots_ref))
    return round(2 * precision * rappel / (precision + rappel), 3)


def detecter_abstention(reponse):
    r = normaliser(reponse)
    return "information_absente" in r.replace(" ", "_") or "information absente" in r or "pas mentionne" in r or "ne mentionne pas" in r or "aucune information" in r


def detecter_signalement_contradiction(reponse):
    r = normaliser(reponse)
    marqueurs = ["contredis", "contradictoire", "en revanche", "cependant", "alors que", "tandis que"]
    return any(m in r for m in marqueurs)


def evaluer_reponse(cas, reponse):
    categorie = cas["categorie"]

    if categorie == "absente":
        correct = detecter_abstention(reponse)
        return {"correct": correct, "f1": None, "abstention_correcte": correct, "contradiction_signalee": None}

    if categorie == "contradictoire":
        signalee = detecter_signalement_contradiction(reponse)
        f1 = f1_mots(cas["reponse_attendue"], reponse)
        return {"correct": signalee, "f1": f1, "abstention_correcte": None, "contradiction_signalee": signalee}

    # repondable / multi_passage
    f1 = f1_mots(cas["reponse_attendue"], reponse)
    return {"correct": f1 >= 0.5, "f1": f1, "abstention_correcte": None, "contradiction_signalee": None}


print("Fonctions de métriques prêtes.")

## Cellule 7 — Boucle d'évaluation : un modèle à la fois

Charge le modèle, évalue tout le jeu de test, mesure la VRAM pic, décharge, passe au suivant.

In [ ]:
resultats_llm = []

for nom_modele, cfg in LLM_CANDIDATS.items():
    print(f"\n{'='*60}")
    print(f"Modèle : {nom_modele} ({cfg['id']}) — quantification {cfg['quant']}")
    print(f"{'='*60}")

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    try:
        t0_chargement = time.perf_counter()
        tokenizer, model = charger_llm(nom_modele)
        temps_chargement_s = time.perf_counter() - t0_chargement
        print(f"  Chargé en {temps_chargement_s:.1f}s")
    except Exception as e:
        print(f"  ❌ Échec du chargement : {e}")
        resultats_llm.append({
            "modele": nom_modele, "id_cas": None, "categorie": None,
            "reponse": None, "erreur_chargement": str(e),
        })
        continue

    for cas in JEU_TEST:
        prompt = construire_prompt(cas["question"], cas["passages"])
        try:
            reponse, temps_ms = generer_reponse(tokenizer, model, cfg, prompt)
            erreur = None
        except Exception as e:
            reponse, temps_ms, erreur = "", None, str(e)

        metriques = evaluer_reponse(cas, reponse) if not erreur else {
            "correct": False, "f1": None, "abstention_correcte": None, "contradiction_signalee": None,
        }

        resultats_llm.append({
            "modele": nom_modele, "id_cas": cas["id"], "categorie": cas["categorie"],
            "question": cas["question"], "reponse": reponse,
            "reponse_attendue": cas["reponse_attendue"],
            "temps_ms": round(temps_ms, 1) if temps_ms else None,
            "erreur": erreur, "precision_chargement": cfg["quant"],
            "temps_chargement_s": round(temps_chargement_s, 1),
            **metriques,
        })

    vram_pic_go = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None
    print(f"  VRAM pic mesurée : {vram_pic_go:.2f} Go" if vram_pic_go else "  (CPU, pas de mesure VRAM)")

    for r in resultats_llm:
        if r["modele"] == nom_modele:
            r["vram_pic_go"] = round(vram_pic_go, 2) if vram_pic_go else None

    decharger_llm(tokenizer, model)
    print(f"  Modèle déchargé, mémoire libérée.")

df_llm = pd.DataFrame(resultats_llm)
print(f"\n{len(df_llm)} évaluations réalisées au total.")

## Cellule 8 — Agrégation par modèle et par catégorie

In [ ]:
df_llm.to_csv(RESULTS_DIR / "llm_resultats_detailles.csv", index=False)

# Taux de bonne réponse par catégorie et par modèle
pivot_categorie = df_llm.pivot_table(
    index="modele", columns="categorie", values="correct", aggfunc="mean"
).round(3)
print("=== Taux de réussite par catégorie (1.0 = parfait) ===")
display(pivot_categorie)

# Résumé global
resume_llm = df_llm.groupby("modele").agg(
    taux_reussite_global=("correct", "mean"),
    f1_moyen=("f1", "mean"),
    temps_ms_moyen=("temps_ms", "mean"),
    vram_pic_go=("vram_pic_go", "first"),
    temps_chargement_s=("temps_chargement_s", "first"),
).round(3)

# Abstention et contradiction séparément (sinon noyées dans la moyenne globale)
abstention = df_llm[df_llm["categorie"] == "absente"].groupby("modele")["abstention_correcte"].mean().round(3)
contradiction = df_llm[df_llm["categorie"] == "contradictoire"].groupby("modele")["contradiction_signalee"].mean().round(3)
resume_llm["taux_abstention_correcte"] = abstention
resume_llm["taux_contradiction_signalee"] = contradiction

resume_llm.to_csv(RESULTS_DIR / "llm_resume.csv")
print("\n=== Résumé global par modèle ===")
display(resume_llm)

## Cellule 9 — Graphiques comparatifs

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

resume_llm["taux_reussite_global"].plot(kind="bar", ax=axes[0], color="#4C72B0", title="Taux de réussite global (plus haut = mieux)")
axes[0].tick_params(axis="x", rotation=30)

resume_llm["taux_abstention_correcte"].plot(kind="bar", ax=axes[1], color="#DD8452", title="Abstention correcte sur cas 'absente' (plus haut = mieux)")
axes[1].tick_params(axis="x", rotation=30)

axes2 = axes[2].twinx()
resume_llm["temps_ms_moyen"].plot(kind="bar", ax=axes[2], color="#55A868", alpha=0.7, position=0, width=0.4)
resume_llm["vram_pic_go"].plot(kind="bar", ax=axes2, color="#C44E52", alpha=0.7, position=1, width=0.4)
axes[2].set_ylabel("Latence moyenne (ms)", color="#55A868")
axes2.set_ylabel("VRAM pic (Go)", color="#C44E52")
axes[2].set_title("Latence vs VRAM")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "graphiques_llm.png", dpi=150)
plt.show()

# Nuage de points : qualité vs latence
plt.figure(figsize=(8, 6))
for nom in resume_llm.index:
    plt.scatter(resume_llm.loc[nom, "temps_ms_moyen"], resume_llm.loc[nom, "taux_reussite_global"], s=150, label=nom)
plt.xlabel("Latence moyenne (ms)")
plt.ylabel("Taux de réussite global")
plt.title("Compromis qualité / vitesse (idéal = en haut à gauche)")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "graphique_compromis_llm.png", dpi=150)
plt.show()

## Cellule 10 — Sélection et sauvegarde du meilleur LLM

Score composite : taux de réussite global (40%), abstention correcte (25% — critique pour éviter les hallucinations), signalement des contradictions (15%), latence inversée (10%), VRAM inversée (10%).

In [ ]:
def normaliser_serie(serie, inverser=False):
    s = serie.dropna()
    if s.empty or s.max() == s.min():
        return pd.Series(0.5, index=serie.index)
    norm = (serie - s.min()) / (s.max() - s.min())
    return 1 - norm if inverser else norm


ponderations = [
    ("taux_reussite_global", 0.40, False),
    ("taux_abstention_correcte", 0.25, False),
    ("taux_contradiction_signalee", 0.15, False),
    ("temps_ms_moyen", 0.10, True),
    ("vram_pic_go", 0.10, True),
]

score = pd.Series(0.0, index=resume_llm.index)
poids_total = pd.Series(0.0, index=resume_llm.index)

for col, poids, inverser in ponderations:
    if col not in resume_llm.columns:
        continue
    dispo = resume_llm[col].notna()
    if not dispo.any():
        continue
    score += normaliser_serie(resume_llm[col], inverser=inverser).fillna(0) * poids
    poids_total += dispo.astype(float) * poids

resume_llm["score_composite"] = (score / poids_total.replace(0, np.nan)).round(4)
resume_trie = resume_llm.sort_values("score_composite", ascending=False)
display(resume_trie[["score_composite", "taux_reussite_global", "taux_abstention_correcte", "temps_ms_moyen", "vram_pic_go"]])

nom_meilleur_llm = resume_trie.index[0]
print(f"\n🏆 Meilleur LLM pour TontoumaBot : {nom_meilleur_llm} (score = {resume_trie.iloc[0]['score_composite']:.3f})")
print(f"   ID Hugging Face : {LLM_CANDIDATS[nom_meilleur_llm]['id']}")
print(f"   Quantification retenue : {LLM_CANDIDATS[nom_meilleur_llm]['quant']}")

In [ ]:
# --- Sauvegarde de la fiche de sélection (le modèle lui-même reste sur le Hub,
#     il est rechargé à la demande avec la même config de quantification) ---
import json as _json

MEILLEUR_LLM_DIR.mkdir(exist_ok=True)

fiche = {
    "nom_modele_source": LLM_CANDIDATS[nom_meilleur_llm]["id"],
    "type": LLM_CANDIDATS[nom_meilleur_llm]["type"],
    "quantification": LLM_CANDIDATS[nom_meilleur_llm]["quant"],
    "score_composite": float(resume_trie.iloc[0]["score_composite"]),
    "taux_reussite_global": float(resume_trie.iloc[0]["taux_reussite_global"]),
    "taux_abstention_correcte": float(resume_trie.iloc[0]["taux_abstention_correcte"]) if pd.notna(resume_trie.iloc[0]["taux_abstention_correcte"]) else None,
    "vram_pic_go": float(resume_trie.iloc[0]["vram_pic_go"]) if pd.notna(resume_trie.iloc[0]["vram_pic_go"]) else None,
    "date_benchmark": pd.Timestamp.now().isoformat(),
}

with open(MEILLEUR_LLM_DIR / "fiche_selection.json", "w", encoding="utf-8") as f:
    _json.dump(fiche, f, ensure_ascii=False, indent=2)

print(f"Fiche de sélection sauvegardée : {MEILLEUR_LLM_DIR / 'fiche_selection.json'}")
print("\nPour recharger ce modèle dans l'application :")
print(f'  charger_llm("{nom_meilleur_llm}")  # réutilise directement la fonction de ce notebook')

## Cellule 11 — Bilan final

In [ ]:
print("=== BILAN DU BENCHMARK LLM ===\n")

print("--- Meilleur par critère individuel ---")
print(f"Meilleur taux de réussite global : {resume_llm['taux_reussite_global'].idxmax()} ({resume_llm['taux_reussite_global'].max():.3f})")
if resume_llm["taux_abstention_correcte"].notna().any():
    print(f"Meilleure abstention (anti-hallucination) : {resume_llm['taux_abstention_correcte'].idxmax()} ({resume_llm['taux_abstention_correcte'].max():.3f})")
print(f"Le plus rapide : {resume_llm['temps_ms_moyen'].idxmin()} ({resume_llm['temps_ms_moyen'].min():.0f} ms)")
if resume_llm["vram_pic_go"].notna().any():
    print(f"Le plus léger en VRAM : {resume_llm['vram_pic_go'].idxmin()} ({resume_llm['vram_pic_go'].min():.2f} Go)")

print(f"\n🏆 MEILLEUR LLM GLOBAL (score composite) : {nom_meilleur_llm}")

print("\n--- Limites de ce run ---")
print(f"- Jeu de test réduit à {len(JEU_TEST)} cas — vise 30 à 50 questions avec au moins 20% de cas 'absente' pour un résultat robuste")
print("- La détection d'abstention/contradiction repose sur des mots-clés (heuristique) — une relecture humaine sur un échantillon reste recommandée")
print("- Le contexte est figé et fourni directement (pas de retrieval réel) — ce notebook isole bien la qualité du LLM, indépendamment du RAG")